# Cross-Modal Face Recognition: Sketch-to-Photo
### Comparative Analysis — resnet ResNet50 vs LightCNN-29

This notebook trains and evaluates two models on the task of matching face sketches to photographs.
Both models are trained with the same pipeline: CUFS pretraining → FS2K fine-tuning using batch-hard triplet loss.

## 1. Configuration

In [ ]:
FS2K_DIR = "/datasets/FS2K/"
CUFS_SKETCH_DIR = "/datasets/CUFS/sketches/"
CUFS_PHOTO_DIR  = "/datasets/CUFS/photos/"
SAVE_RESNET    = "/resnet_checkpoint.pth"
SAVE_VGG  = "/vgg_checkpoint.pth"

## 2. Imports

In [ ]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)
print('Torch:', torch.__version__)

## 3. Data Loading

**CUFS** uses a filename-based mapping (e.g. `f-039-sz1.jpg` → `f-039.jpg`).  
**FS2K** uses a folder structure with `photo1/2/3` and `sketch1/2/3` subfolders.

In [ ]:
def map_sketch_to_photo(sketch_id):
    sketch_id = sketch_id.replace('.jpg', '').replace('-sz1', '')
    if sketch_id.startswith('F2-'):  return sketch_id.replace('F2-', 'f-')
    if sketch_id.startswith('f1-') or sketch_id.startswith('f-'): return sketch_id
    if sketch_id.startswith('M2-'):  return sketch_id.replace('M2-', 'm-')
    if sketch_id.startswith('m1-') or sketch_id.startswith('m-'): return sketch_id
    return None


class CUFSDataset(Dataset):
    def __init__(self, sketch_dir, photo_dir, transform=None):
        self.sketch_dir = sketch_dir
        self.photo_dir  = photo_dir
        self.transform  = transform
        photo_map = {p.replace('.jpg', ''): p for p in os.listdir(photo_dir)}
        self.pairs = []
        for s in os.listdir(sketch_dir):
            pid = map_sketch_to_photo(s)
            if pid and pid in photo_map:
                self.pairs.append((s, photo_map[pid]))
        print(f'CUFS: {len(self.pairs)} pairs found')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        sn, pn = self.pairs[idx]
        sketch = Image.open(os.path.join(self.sketch_dir, sn)).convert('RGB')
        photo  = Image.open(os.path.join(self.photo_dir,  pn)).convert('RGB')
        if self.transform:
            sketch = self.transform(sketch)
            photo  = self.transform(photo)
        return sketch, photo, idx


class FS2KDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.pairs = []
        for split in ['1', '2', '3']:
            pd = os.path.join(root_dir, 'photo',  f'photo{split}')
            sd = os.path.join(root_dir, 'sketch', f'sketch{split}')
            if not os.path.exists(pd) or not os.path.exists(sd):
                continue
            for fname in sorted(os.listdir(pd)):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
                sp = os.path.join(sd, fname.replace('image', 'sketch'))
                if os.path.exists(sp):
                    self.pairs.append((sp, os.path.join(pd, fname)))
        print(f'FS2K: {len(self.pairs)} pairs found')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        sp, pp = self.pairs[idx]
        sketch = Image.open(sp).convert('RGB')
        photo  = Image.open(pp).convert('RGB')
        if self.transform:
            sketch = self.transform(sketch)
            photo  = self.transform(photo)
        return sketch, photo, idx

## 4. Preprocessing

Images are resized to 112×112 and normalized to [-1, 1]. Training sketches receive augmentation (flip, brightness, rotation) to reduce overfitting on the small dataset.

In [ ]:
def pil_to_tensor(img):
    """Convert PIL image to float tensor without numpy/torchvision."""
    img = img.resize((112, 112), Image.BILINEAR).convert('RGB')
    pixels = list(img.getdata())
    r = torch.tensor([p[0] for p in pixels], dtype=torch.float32).reshape(112, 112)
    g = torch.tensor([p[1] for p in pixels], dtype=torch.float32).reshape(112, 112)
    b = torch.tensor([p[2] for p in pixels], dtype=torch.float32).reshape(112, 112)
    return (torch.stack([r, g, b]) / 127.5) - 1.0

def transform_train(img):
    img = img.resize((112, 112), Image.BILINEAR).convert('RGB')
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        f = random.uniform(0.7, 1.3)
        img = img.point(lambda p: min(255, int(p * f)))
    if random.random() > 0.5:
        img = img.rotate(random.uniform(-15, 15))
    pixels = list(img.getdata())
    r = torch.tensor([p[0] for p in pixels], dtype=torch.float32).reshape(112, 112)
    g = torch.tensor([p[1] for p in pixels], dtype=torch.float32).reshape(112, 112)
    b = torch.tensor([p[2] for p in pixels], dtype=torch.float32).reshape(112, 112)
    return (torch.stack([r, g, b]) / 127.5) - 1.0

def transform_test(img):
    return pil_to_tensor(img)

# Build datasets
cufs_ds      = CUFSDataset(CUFS_SKETCH_DIR, CUFS_PHOTO_DIR, transform_train)
print(f'Total training pairs: CUFS={len(cufs_ds)}')

random.seed(42)  # reproducible split

full_train = FS2KDataset(FS2K_DIR, transform_train)
full_test  = FS2KDataset(FS2K_DIR, transform_test)

N       = len(full_train)
indices = list(range(N))
random.shuffle(indices)

split      = int(0.8 * N)
train_idx  = indices[:split]
test_idx   = indices[split:]

from torch.utils.data import Subset
fs2k_ds_train = Subset(full_train, train_idx)
fs2k_ds_test  = Subset(full_test,  test_idx)

print(f'FS2K Train: {len(fs2k_ds_train)} | Test: {len(fs2k_ds_test)}')

## 5. Loss Function — Batch-Hard Triplet Loss

For each sketch (anchor), the hardest negative (most similar wrong photo) is selected within the batch. This ensures the model always trains on informative examples.

In [ ]:
def batch_hard_triplet_loss(sketch_embs, photo_embs, margin=0.3):
    B   = sketch_embs.size(0)
    sim = torch.mm(sketch_embs, photo_embs.t())
    total_loss = torch.tensor(0.0, device=sketch_embs.device, requires_grad=True)
    count = 0
    for i in range(B):
        pos_sim  = sim[i, i]
        neg_sims = torch.cat([sim[i, :i], sim[i, i+1:]])
        hard_neg = neg_sims.max()
        triplet  = margin - pos_sim + hard_neg
        if triplet.item() > 0:
            total_loss = total_loss + triplet
            count += 1
    if count == 0:
        return torch.tensor(0.0, device=sketch_embs.device, requires_grad=True)
    return total_loss / count


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for sketches, photos, _ in loader:
        sketches, photos = sketches.to(device), photos.to(device)
        s_emb = F.normalize(model(sketches), p=2, dim=1)
        p_emb = F.normalize(model(photos),   p=2, dim=1)
        loss  = batch_hard_triplet_loss(s_emb, p_emb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

## 6. Evaluation Function

Reports four standard metrics used in heterogeneous face recognition literature:
- **Rank-1 / Rank-5 / Rank-10**: retrieval accuracy at different cutoffs
- **ROC AUC**: overall separability between genuine and impostor pairs
- **TAR@FAR=1%**: true accept rate at a 1% false accept rate operating point

In [ ]:
def evaluate(model, dataset, model_name):
    model.eval()
    all_s, all_p = [], []
    with torch.no_grad():
        for sketches, photos, _ in DataLoader(dataset, batch_size=32, num_workers=0):
            all_s.append(F.normalize(model(sketches.to(device)), dim=1))
            all_p.append(F.normalize(model(photos.to(device)),   dim=1))
    all_s = torch.cat(all_s)
    all_p = torch.cat(all_p)
    N     = all_s.size(0)

    sim_matrix = torch.mm(all_s, all_p.t())
    ranks      = sim_matrix.argsort(dim=1, descending=True)
    correct    = torch.arange(N, device=device)

    rank1  = (ranks[:, 0] == correct).float().mean().item()
    rank5  = sum(correct[i].item() in ranks[i, :5].tolist()  for i in range(N)) / N
    rank10 = sum(correct[i].item() in ranks[i, :10].tolist() for i in range(N)) / N

    sims_flat   = sim_matrix.cpu().flatten().tolist()
    gt_flat     = [1 if i == j else 0 for i in range(N) for j in range(N)]
    auc         = roc_auc_score(gt_flat, sims_flat)
    fpr, tpr, _ = roc_curve(gt_flat, sims_flat)
    tar_far1    = float(tpr[next(i for i, f in enumerate(fpr) if f >= 0.01)])

    print(f'\n{"="*45}')
    print(f'  {model_name} — Results on FS2K ({N} pairs)')
    print(f'{"="*45}')
    print(f'  Rank-1  Accuracy : {rank1*100:6.2f}%')
    print(f'  Rank-5  Accuracy : {rank5*100:6.2f}%')
    print(f'  Rank-10 Accuracy : {rank10*100:6.2f}%')
    print(f'  ROC AUC          : {auc:.4f}')
    print(f'  TAR@FAR=1%       : {tar_far1*100:6.2f}%')
    print(f'{"="*45}')
    return {'model': model_name, 'rank1': rank1, 'rank5': rank5,
            'rank10': rank10, 'auc': auc, 'tar_far1': tar_far1}

---
## 7. Model A — resnet ResNet50

ResNet50 pretrained on ImageNet, with a 512-d projection head. The backbone provides strong general visual features; the head is fine-tuned to produce face embeddings.

**Training strategy:**
1. Phase 1 (CUFS) — freeze backbone, train head only at lr=1e-3
2. Phase 2 (FS2K) — unfreeze last ResNet block, train with lower lr
3. Phase 3 (FS2K extended) — unfreeze more layers, train with augmentation at very low lr

In [ ]:
import torchvision.models as tvm

class ResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = tvm.resnet50(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.head = nn.Linear(2048, 512)

    def forward(self, x):
        return self.head(self.backbone(x).flatten(1))

model = ResNet50().to(device)

# load checkpoint if it exists
if os.path.exists(SAVE_RESNET):
    model.load_state_dict(torch.load(SAVE_RESNET, map_location=device))
    print('Loaded existing resnet checkpoint')
else:
    print('resnet: starting from ImageNet weights')

print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Phase 1: CUFS pretraining (head only) 
for p in model.backbone.parameters(): p.requires_grad = False
for p in model.head.parameters():     p.requires_grad = True

cufs_loader = DataLoader(cufs_ds, batch_size=8, shuffle=True, num_workers=0)
optimizer   = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

print('=== RESNET-50 Phase 1: CUFS pretraining ===')
for epoch in range(1, 11):  # 10 epochs
    start = time.time()
    loss  = train_one_epoch(model, cufs_loader, optimizer)
    print(f'  Epoch {epoch:02d}/10 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)

In [ ]:
# Phase 2: FS2K fine-tuning (unfreeze layer4 + head) 
for p in model.backbone[-2].parameters(): p.requires_grad = True

optimizer = torch.optim.Adam([
    {'params': model.backbone[-2].parameters(), 'lr': 1e-5},
    {'params': model.head.parameters(),         'lr': 1e-4},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== resnet Phase 2: FS2K fine-tuning ===')
for epoch in range(1, 21):  # 20 epochs
    start = time.time()
    loss  = train_one_epoch(model, fs2k_loader, optimizer)
    print(f'  Epoch {epoch:02d}/20 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)

In [ ]:
# Phase 3: Extended FS2K training (unfreeze more, very low lr) 
for p in model.backbone[-4:].parameters(): p.requires_grad = True

optimizer = torch.optim.Adam([
    {'params': model.backbone[-4:].parameters(), 'lr': 5e-6},
    {'params': model.head.parameters(),          'lr': 5e-5},
])

print('=== resnet Phase 3: Extended FS2K training ===')
for epoch in range(1, 31):  # 30 epochs
    start = time.time()
    loss  = train_one_epoch(model, fs2k_loader, optimizer)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)
    if loss < 0.01:  # stop early if converged
        print('  Converged — stopping early')
        break

In [ ]:
# Evaluate resnet
model.load_state_dict(torch.load(SAVE_RESNET, map_location=device))
results_resnet = evaluate(model, fs2k_ds_test, 'resnet ResNet50')

---
## 8. Model B — VGG-16

VGG-16 uses Max Feature Map (MFM) activations instead of ReLU, which suppress noisy activations and select the most discriminative features. It was specifically designed for heterogeneous face recognition and is a standard baseline in sketch-photo literature.


In [ ]:
#VGG16 Baseline 
class VGGBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tvm.vgg16(weights='IMAGENET1K_V1')
        self.backbone = vgg.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Linear(512, 512)

    def forward(self, x):
        x = self.pool(self.backbone(x)).flatten(1)
        return self.head(x)

vgg_model = VGGBaseline().to(device)

if os.path.exists(SAVE_VGG):
    vgg_model.load_state_dict(torch.load(SAVE_VGG, map_location=device))
    print('Loaded existing VGG checkpoint')
else:
    print('VGG16: starting from ImageNet weights')

# freeze backbone, train head only for phase 1
for p in vgg_model.backbone.parameters(): p.requires_grad = False
for p in vgg_model.head.parameters():     p.requires_grad = True

print(f'Trainable params: {sum(p.numel() for p in vgg_model.parameters() if p.requires_grad):,}')

In [ ]:
cufs_loader = DataLoader(cufs_ds, batch_size=8, shuffle=True, num_workers=0)
optimizer_v = torch.optim.Adam(filter(lambda p: p.requires_grad, vgg_model.parameters()), lr=1e-3)

print('=== VGG16 Phase 1: CUFS pretraining ===')
for epoch in range(1, 11):
    start = time.time()
    loss  = train_one_epoch(vgg_model, cufs_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/10 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)

In [ ]:
# unfreeze last few VGG conv layers
for p in vgg_model.backbone[-6:].parameters(): p.requires_grad = True

optimizer_v = torch.optim.Adam([
    {'params': vgg_model.backbone[-6:].parameters(), 'lr': 1e-5},
    {'params': vgg_model.head.parameters(),          'lr': 1e-4},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== VGG16 Phase 2: FS2K fine-tuning ===')
for epoch in range(1, 21):
    start = time.time()
    loss  = train_one_epoch(vgg_model, fs2k_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)
    if loss < 0.01:
        print('  Converged — stopping early')
        break

In [ ]:
# unfreeze more VGG layers for phase 3
for p in vgg_model.backbone[-10:].parameters():
    p.requires_grad = True

optimizer_v = torch.optim.Adam([
    {'params': vgg_model.backbone[-10:].parameters(), 'lr': 5e-6},
    {'params': vgg_model.head.parameters(),           'lr': 5e-5},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== VGG16 Phase 3: Extended FS2K training ===')
for epoch in range(1, 31):
    start = time.time()
    loss  = train_one_epoch(vgg_model, fs2k_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)
    if loss < 0.01:
        print('  Converged — stopping early')
        break

In [ ]:
vgg_model.load_state_dict(torch.load(SAVE_VGG, map_location=device))
results_vgg = evaluate(vgg_model, fs2k_ds_test, 'VGG16 Baseline')

---
## 9. Comparison Table

In [ ]:
print('\n' + '='*55)
print(f'{"Metric":<20} {"resnet ResNet50":>16} {"VGG Baseline":>14}')
print('='*55)
for key, label in [
    ('rank1',    'Rank-1'),
    ('rank5',    'Rank-5'),
    ('rank10',   'Rank-10'),
    ('auc',      'ROC AUC'),
    ('tar_far1', 'TAR@FAR=1%'),
]:
    a = results_resnet[key] * 100
    l = results_vgg[key] * 100
    winner = '<' if a > l else '>'
    print(f'{label:<20} {a:>15.2f}% {l:>13.2f}%  {winner}')
print('='*55)
print('< = ResNet-50 wins   > = VGG wins')

In [ ]:
def query_sketch(sketch_path, model, dataset, top_k=5):
    model.eval()

    query       = Image.open(sketch_path).convert('RGB')
    query_tensor = transform_test(query).unsqueeze(0).to(device)

    with torch.no_grad():
        query_emb = F.normalize(model(query_tensor), p=2, dim=1)

    # build photo gallery
    all_p = []
    with torch.no_grad():
        for _, photos, _ in DataLoader(dataset, batch_size=32, num_workers=0):
            all_p.append(F.normalize(model(photos.to(device)), dim=1))
    all_p = torch.cat(all_p)
    sims        = torch.mm(query_emb, all_p.t()).squeeze(0)
    top_indices = sims.argsort(descending=True)[:top_k]
    fig, axes = plt.subplots(1, top_k + 1, figsize=((top_k + 1) * 3, 4))

    axes[0].imshow(query.resize((112, 112)))
    axes[0].set_title('Query Sketch', fontsize=9)
    axes[0].axis('off')

    for rank, idx in enumerate(top_indices):
        idx        = idx.item()
        photo_path = dataset.dataset.pairs[dataset.indices[idx]][1]
        score      = sims[idx].item()
        photo      = Image.open(photo_path).convert('RGB').resize((112, 112))
        axes[rank + 1].imshow(photo)
        axes[rank + 1].set_title(f'Rank {rank+1}\n{score:.3f}', fontsize=9)
        axes[rank + 1].axis('off')

    plt.suptitle(f'Top {top_k} matches — {os.path.basename(sketch_path)}', fontsize=11)
    plt.tight_layout()
    plt.savefig('query_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to query_result.png')
